<a href="https://colab.research.google.com/github/rainforest01-coder/ESAA_files/blob/OB/week2%EC%88%98%EC%83%81%EC%9E%91_%EC%88%98%EB%A3%8C%EC%98%88%EC%B8%A12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 데이콘 x BDA 제 2회 학습자 수료 예측 AI 경진대회

-데이터 특징
> 혼합형 데이터, 결측치 매우 많음, F1스코어( 확률 예측 자체보다 0/1 결정 임계값이 성능 좌우), 작은 데이터 과적합 위험

1. 코드흐름 요약
* 전처리
> 복잡 대비 효과 낮은 칼럼 제거, 희소범주는 기타로 통합해 원핫인코딩,
LR용 OHE 피처(고정 피처 셋)와 CatBoost용 범주형 피처를 분리해 전처리를 정리
* Modeling
F1은 FN을 줄일수록 유리하다고 보고, 기본 예측을 전부 1로 둔 뒤 1일 확률이 낮은 순서로 고정 비율만 0으로 뒤집는(flip-rate 고정) 전략을 사용,

평가 데이터 확률이 특정 구간에 촘촘히 몰려 있어 threshold 기반보다 비율 고정이 더 안정적,

여러 모델(LR, XGB, LGBM, RF, SVM 등)을 비교한 결과, LR이 LB가 가장 높고 확률 분산도 커 Base Model로 채택

 - Stage 1 — Rescue/Demote (CatBoost)

CatBoost는 LR의 OOF/Test 확률(lr_oof_prob)을 입력 피처로 포함해 LR 예측을 보정  
 또한 flip 경계 근처 샘플에 가중치를 부여해 경계 판단을 강화  
LR이 0으로 뒤집은 샘플 중 CB가 1 가능성이 높다고 본 샘플은 Rescue(0→1) 하고, LR이 1로 둔 샘플 중 CB가 1 가능성이 낮다고 본 샘플은 Demote(1→0) 하여 예측을 교정


- Stage 2 — Multi-CB Correction  
서로 다른 목표로 튜닝된 4개 CatBoost를 조합해 Stage 1 결과를 추가 보정

Rescue는 A/B의 교집합으로 보수적으로 수행하고, Demote는 A/B/C를 결합해 확대하되 C11 기반 veto와 추가 규칙으로 과도한 Demote를 통제



2. 새롭게 알게된 점

-LR용 OHE 피처(고정 피처 셋)와 CatBoost용 범주형 피처를 분리해 전처리를 정리해두어야한다.
-1로 예측하는 비율 고정법을 알게 됨
-기존의 앙상블, 스태킹 외에 여러 모델을 사용하는 예측법을 알게 됨


In [ ]:
def all1_flip_pred(prob: np.ndarray, flip_rate: float):
    prob = np.asarray(prob).reshape(-1)
    n = len(prob)
    k = int(round(n * float(flip_rate)))
    k = max(0, min(k, n))
    pred = np.ones(n, dtype=int)
    if k > 0:
        order = np.argsort(prob, kind="mergesort")
        pred[order[:k]] = 0
    return pred

    ##확률이 낮은 순서로 일정비율만큼 0으로 뒤집기